# Qwen3-Embedding-0.6B — Fine-tuning MTEB French avec Unsloth

**Objectif :** Fine-tuner `Qwen/Qwen3-Embedding-0.6B` sur MIRACL-fr avec CachedMNR Loss.  
**Stack :** Unsloth (1.8–3.3x plus rapide) + SentenceTransformers + LoRA  
**Hardware :** RTX 3070 8GB  

### Pourquoi Qwen3-Embedding-0.6B ?
- ✅ Officiellement supporté par Unsloth (modules.json natif, pooling correct out-of-the-box)
- ✅ Score MTEB(fra) baseline ~68 — meilleur que F2LLM-330M dans sa catégorie de taille
- ✅ Même backbone Qwen3 que F2LLM mais avec pooling natif — pas de debug pooling nécessaire
- ✅ Mirror Unsloth officiel : `unsloth/Qwen3-Embedding-0.6B`

## 0. Vérification GPU

In [1]:
# CELLULE 0 — à exécuter EN PREMIER avant tout import
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f'CUDA disponible : {torch.cuda.is_available()}')
print(f'GPU             : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN"}')
print(f'PyTorch         : {torch.__version__}')
assert torch.cuda.is_available(), 'GPU non détecté — vérifier le kernel Python'


CUDA disponible : True
GPU             : NVIDIA GeForce RTX 3070
PyTorch         : 2.12.1+cu130


## 1. Imports & vérifications

In [2]:
import os
import sys
import gc
import subprocess
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from datasets import load_dataset
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

# Unsloth — doit être importé EN PREMIER pour patcher transformers
from unsloth import FastSentenceTransformer

import unsloth
import sentence_transformers
print(f'Unsloth             : {unsloth.__version__}')
print(f'SentenceTransformers: {sentence_transformers.__version__}')
print(f'PyTorch             : {torch.__version__}')

DEVICE = 'cuda'

def get_gpu_temp():
    try:
        r = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=temperature.gpu', '--format=csv,noheader'],
            encoding='utf-8'
        )
        return r.strip() + '°C'
    except:
        return 'N/A'

def get_vram():
    return f'{torch.cuda.memory_allocated() / 1e9:.2f} GB'

print(f'GPU temp : {get_gpu_temp()}')
print(f'VRAM     : {get_vram()}')


W0703 09:50:22.781000 7084 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0703 09:50:23.038000 7084 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
C:\Users\aurel\AppData\Local\Temp\ipykernel_7084\3077087729.py:12: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, losses
f:\codeAI\llm\.venv\lib\site-packages\unsloth\__init__.py:153: UserWarning: WARNING: Unslot

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


f:\codeAI\llm\.venv\lib\site-packages\unsloth_zoo\temporary_patches\moe_utils.py:180: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  x = torch.ones((1, 8), device=device, dtype=dtype)


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


Unsloth             : 2026.6.9
SentenceTransformers: 5.6.0
PyTorch             : 2.12.1+cu130
GPU temp : 49°C
VRAM     : 0.01 GB


## 2. Chargement du modèle — Qwen3-Embedding-0.6B

Contrairement à F2LLM, Qwen3-Embedding-0.6B est **officiellement supporté** par Unsloth :  
- `modules.json` natif → pooling EOS correct out-of-the-box  
- Pas de `F2LLMPooling` custom nécessaire  
- LoRA injecté automatiquement  
- Mirror Unsloth : `unsloth/Qwen3-Embedding-0.6B` (chargement optimisé)

In [3]:
# MODEL_NAME  = 'unsloth/Qwen3-Embedding-0.6B'  # mirror Unsloth officiel

MODEL_NAME = "intfloat/multilingual-e5-base"

MAX_SEQ_LEN = 256
LOAD_4BIT   = True   # QLoRA : ~2GB VRAM pour le modèle

model = FastSentenceTransformer.from_pretrained(
    MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,        # bfloat16 auto
    load_in_4bit   = True,
    load_in_8bit   = False,
    load_in_16bit  = False,
)

print(model)
print(f'\nVRAM après chargement : {get_vram()}')
print(f'GPU temp              : {get_gpu_temp()}')


Unsloth: Using fast encoder path for xlm-roberta with 4-bit quantization + SDPA


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

VRAM après chargement : 0.82 GB
GPU temp              : 49°C


## 2b. Vérification du pooling

Qwen3-Embedding a un pooling natif — on vérifie que les embeddings sont corrects  
en comparant avec la référence HuggingFace Transformers.

In [4]:
# Vérification rapide des embeddings
test_texts = [
    'Quelle est la capitale de la France ?',
    'Paris est la capitale de la France.',
    'Le bœuf bourguignon est un plat traditionnel.'
]

emb = model.encode(test_texts, convert_to_tensor=True)

print(f'Shape  : {emb.shape}')           # (3, 1024) — Qwen3-0.6B fait 1024 dims
print(f'Normes : {emb.norm(dim=1)}')     # [1.0, 1.0, 1.0]
print(f'dtype  : {emb.dtype}')

sim_pos = (emb[0] @ emb[1]).item()
sim_neg = (emb[0] @ emb[2]).item()
print(f'\nSim query–positif : {sim_pos:.4f}')
print(f'Sim query–négatif : {sim_neg:.4f}')
print(f'Margin            : {sim_pos - sim_neg:.4f}')
assert sim_pos > sim_neg, 'Le modèle de base doit déjà ranker correctement'



Shape  : torch.Size([3, 768])
Normes : tensor([1., 1., 1.], device='cuda:0')
dtype  : torch.float32

Sim query–positif : 0.9253
Sim query–négatif : 0.7617
Margin            : 0.1636


In [5]:
# ── Injection LoRA manuelle — OBLIGATOIRE avant fit() ─────────────────────────
# FastSentenceTransformer en 4bit ne l'injecte pas automatiquement via fit()
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r              = 16,
    lora_alpha     = 32,
    target_modules = [
        "query",    # XLM-RoBERTa utilise query/key/value
        "key",      # (pas q_proj/k_proj/v_proj comme les LLMs décodeurs)
        "value",
        "dense"     # FFN
    ],
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = TaskType.FEATURE_EXTRACTION
)

model[0].auto_model = get_peft_model(model[0].auto_model, lora_config)

# Attendu : ~1-2% — ~10M params entraînables

# Vérification
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nTrainable : {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")
assert trainable > 0, "LoRA non injecté — vérifier model[0].auto_model"
print("✅ LoRA injecté")


Trainable : 2,678,784 / 237,960,192  (1.13%)
✅ LoRA injecté


## 3. Dataset — MIRACL French (complet)

232k paires (anchor, positive) en français. On utilise `fr-triplet-all`.

In [6]:
ds = load_dataset(
    'sentence-transformers/miracl',
    'fr-triplet-all',
    split='train'
)

ds = ds.select(range(100_000))
print(f'Exemples  : {len(ds)}')
print(f'Colonnes  : {ds.column_names}')
print()
print('anchor  :', ds[0]['anchor'][:100])
print('positive:', ds[0]['positive'][:100])


Exemples  : 100000
Colonnes  : ['anchor', 'positive', 'negative']

anchor  : Qui était Princesse Diana?
positive: Diana Spencer
La « dianamania » bat son plein les premières années qui suivent la mort de la princes


## 4. Conversion en InputExample pour SentenceTransformers

In [7]:
train_examples = [
    InputExample(texts=[row['anchor'], row['positive']])
    for row in ds
]

print(f'Exemples MNR : {len(train_examples)}')
print(f'Exemple[0]   : {train_examples[0].texts[0][:80]}')


Exemples MNR : 100000
Exemple[0]   : Qui était Princesse Diana?


## 5. Loss — CachedMultipleNegativesRankingLoss

Calcule les embeddings en mini-chunks de 8, puis la loss sur le batch complet (64).  
Résultat : 63 négatifs in-batch par anchor avec seulement 8 séquences en VRAM à la fois.

In [8]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_examples,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = 0,     # Windows
    pin_memory  = False
)

train_loss = losses.CachedMultipleNegativesRankingLoss(
    model,
    mini_batch_size   = 8,
    show_progress_bar = False
)

print(f'Dataset         : {len(train_examples)} exemples')
print(f'Batch size      : {BATCH_SIZE}')
print(f'Batches / epoch : {len(train_loader)}')
print(f'Négatifs / step : {BATCH_SIZE - 1}')


Dataset         : 100000 exemples
Batch size      : 64
Batches / epoch : 1563
Négatifs / step : 63


## 6. Paramètres d'entraînement

In [9]:
NUM_EPOCHS  = 3
LR          = 2e-5
WARMUP_FRAC = 0.1
OUTPUT_PATH = './e5-embedding-finetuned-fr'

warmup_steps = int(len(train_loader) * NUM_EPOCHS * WARMUP_FRAC)

print(f'Epochs        : {NUM_EPOCHS}')
print(f'Batches/epoch : {len(train_loader)}')
print(f'Total steps   : {len(train_loader) * NUM_EPOCHS}')
print(f'Warmup steps  : {warmup_steps}')
print(f'LR            : {LR}')
print(f'VRAM avant    : {get_vram()}')
print(f'GPU temp      : {get_gpu_temp()}')


Epochs        : 3
Batches/epoch : 1563
Total steps   : 4689
Warmup steps  : 468
LR            : 2e-05
VRAM avant    : 0.83 GB
GPU temp      : 49°C


## 7. Entraînement

`fit()` gère nativement : progress bar ETA ✅ · logging loss ✅ · warmup + scheduler ✅ · checkpoints ✅

In [10]:
# Désactiver torch.dynamo — incompatible avec LoRA + CachedMNR
import torch._dynamo
torch._dynamo.disable()
#torch._dynamo.config.suppress_errors = True
#torch._dynamo.reset()

# Augmenter la limite de recompilation
#torch._dynamo.config.recompile_limit = 64

#print("✅ Dynamo recompile limit augmentée")

In [11]:
print('🚀 Démarrage entraînement Unsloth + multilingual-e5-base.')
print()

model.fit(
    train_objectives      = [(train_loader, train_loss)],
    epochs                = NUM_EPOCHS,
    warmup_steps          = warmup_steps,
    optimizer_params      = {'lr': LR},
    output_path           = OUTPUT_PATH,
    save_best_model       = True,
    show_progress_bar     = True,
    checkpoint_path       = './checkpoints',
    checkpoint_save_steps = len(train_loader),  # checkpoint chaque epoch
)

print(f'\n✅ Entraînement terminé')
print(f'VRAM après : {get_vram()}')
print(f'GPU temp   : {get_gpu_temp()}')
print(f'Modèle     : {OUTPUT_PATH}')


🚀 Démarrage entraînement Unsloth + multilingual-e5-base.



Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.867311
1000,0.147571
1500,0.132973
2000,0.129380
2500,0.127676
3000,0.130708
3500,0.125860
4000,0.127581
4500,0.132730


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Entraînement terminé
VRAM après : 0.84 GB
GPU temp   : 53°C
Modèle     : ./e5-embedding-finetuned-fr


## 8. Sauvegarde

- `save_pretrained()` → adaptateurs LoRA seuls (~15 MB)  
- `save_pretrained_merged()` → modèle complet mergé (~1.2 GB) prêt pour MTEB

In [12]:
LORA_PATH   = './e5-embedding-lora-adapters'
MERGED_PATH = './e5-embedding-merged'

os.makedirs(LORA_PATH,   exist_ok=True)
os.makedirs(MERGED_PATH, exist_ok=True)

model.save_pretrained(LORA_PATH)
print(f'✅ LoRA adapters : {LORA_PATH}')

model.save_pretrained_merged(MERGED_PATH)
print(f'✅ Modèle mergé  : {MERGED_PATH}')

for p in [LORA_PATH, MERGED_PATH]:
    if os.path.exists(p):
        total = sum(
            os.path.getsize(os.path.join(p, f))
            for f in os.listdir(p)
            if os.path.isfile(os.path.join(p, f))
        )
        print(f'  {p:<45} {total/1e6:.0f} MB')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ LoRA adapters : ./e5-embedding-lora-adapters


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modèle mergé  : ./e5-embedding-merged
  ./e5-embedding-lora-adapters                  842 MB
  ./e5-embedding-merged                         842 MB


## 9. Vérification qualitative post-training

In [13]:
# Charger le modèle fine-tuné pour l'évaluation
model_eval = FastSentenceTransformer.from_pretrained(
    MERGED_PATH,
    for_inference  = True,   # OBLIGATOIRE pour l'inférence avec Unsloth
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = False,  # merge = full precision pour l'eval
    load_in_8bit   = False,
    load_in_16bit  = True,
)

pairs = [
    (
        'Quelle est la capitale de la France ?',
        'Paris est la capitale et la plus grande ville de France.',
        'Le bœuf bourguignon est un plat traditionnel français.'
    ),
    (
        'Comment fonctionne un moteur électrique ?',
        "Un moteur électrique convertit l'énergie électrique en énergie mécanique.",
        "Les voitures thermiques utilisent de l'essence ou du diesel."
    ),
    (
        'Quels sont les symptômes du diabète ?',
        'Les symptômes incluent une soif excessive, une fatigue et des urinations fréquentes.',
        'Le Tour de France est une course cycliste annuelle.'
    ),
    (
        'Qui a écrit Les Misérables ?',
        'Les Misérables est un roman de Victor Hugo publié en 1862.',
        'La recette de la tarte tatin utilise des pommes caramélisées.'
    ),
]

print(f'{"Query":<45} {"sim(+)":>7} {"sim(-)": >7} {"margin":>8}')
print('-' * 75)

for query, pos, neg in pairs:
    eq = model_eval.encode([query], convert_to_tensor=True)
    ep = model_eval.encode([pos],   convert_to_tensor=True)
    en = model_eval.encode([neg],   convert_to_tensor=True)

    sim_pos = (eq @ ep.T).item()
    sim_neg = (eq @ en.T).item()
    margin  = sim_pos - sim_neg

    ok = '✅' if margin > 0 else '❌'
    print(f'{ok} {query[:43]:<45} {sim_pos:>7.4f} {sim_neg:>7.4f} {margin:>8.4f}')


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: ./e5-embedding-merged
Key                                                                 | Status     | 
--------------------------------------------------------------------+------------+-
encoder.layer.{0...11}.intermediate.dense.base_layer.weight         | UNEXPECTED | 
encoder.layer.{0...11}.attention.self.value.lora_B.default.weight   | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.base_layer.weight               | UNEXPECTED | 
encoder.layer.{0...11}.attention.output.dense.lora_A.default.weight | UNEXPECTED | 
encoder.layer.{0...11}.attention.self.value.base_layer.weight       | UNEXPECTED | 
encoder.layer.{0...11}.output.dense.lora_A.default.weight           | UNEXPECTED | 
encoder.layer.{0...11}.intermediate.dense.lora_B.default.weight     | UNEXPECTED | 
encoder.layer.{0...11}.attention.self.key.base_layer.bias           | UNEXPECTED | 
encoder.layer.{0...11}.attention.self.key.lora_A.default.weight     | UNEXPECTED | 
encoder.layer.{0...1

NotImplementedError: "normal_kernel_cuda" not implemented for 'Byte'

## 11. Évaluation MTEB(fra) — benchmark officiel

Évaluation tâche par tâche avec sauvegarde immédiate — reprend là où on s'est arrêté.

In [2]:
import bitsandbytes as bnb
import torch
from safetensors.torch import load_file
from transformers import AutoModel, AutoTokenizer
import os

EVAL_PATH = "./e5-embedding-eval-clean"
os.makedirs(EVAL_PATH, exist_ok=True)

saved_weights = load_file("./e5-embedding-merged/model.safetensors", device="cpu")

# ── Regrouper les composants de quantization par couche ─────────────────────
def find_quant_layers(weights):
    """Trouve toutes les couches quantifiées NF4 et regroupe leurs composants."""
    layers = {}
    for key in weights:
        if key.endswith(".base_layer.weight"):
            prefix = key.replace(".weight", "")
            layers[prefix] = {
                "weight": weights[key],
                "absmax": weights.get(f"{prefix}.weight.absmax"),
                "quant_state": weights.get(f"{prefix}.weight.quant_state.bitsandbytes__nf4"),
                "nested_absmax": weights.get(f"{prefix}.weight.nested_absmax"),
                "nested_quant_map": weights.get(f"{prefix}.weight.nested_quant_map"),
                "quant_map": weights.get(f"{prefix}.weight.quant_map"),
            }
    return layers

quant_layers = find_quant_layers(saved_weights)
print(f"Couches quantifiées trouvées : {len(quant_layers)}")

# Vérifier ce qu'on a exactement pour une couche
sample_key = list(quant_layers.keys())[0]
print(f"\nExemple — {sample_key} :")
for k, v in quant_layers[sample_key].items():
    if v is not None:
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: None")

Couches quantifiées trouvées : 73

Exemple — encoder.layer.0.attention.output.dense.base_layer :
  weight: shape=torch.Size([294912, 1]), dtype=torch.uint8
  absmax: shape=torch.Size([9216]), dtype=torch.uint8
  quant_state: shape=torch.Size([168]), dtype=torch.uint8
  nested_absmax: shape=torch.Size([36]), dtype=torch.float32
  nested_quant_map: shape=torch.Size([256]), dtype=torch.float32
  quant_map: shape=torch.Size([16]), dtype=torch.float32


In [3]:
import torch
import bitsandbytes as bnb
from bitsandbytes.nn import Params4bit
from safetensors.torch import load_file
from transformers import AutoModel, AutoTokenizer
import os

EVAL_PATH = "./e5-embedding-eval-clean"
os.makedirs(EVAL_PATH, exist_ok=True)

saved_weights = load_file("./e5-embedding-merged/model.safetensors", device="cpu")

def dequant_layer(prefix, weights):
    """Reconstruit et déquantifie un poids NF4 depuis ses composants sauvegardés."""
    qweight = weights[f"{prefix}.weight"]                                    # uint8 data
    quant_state_bytes = weights.get(f"{prefix}.weight.quant_state.bitsandbytes__nf4")
    absmax = weights.get(f"{prefix}.weight.absmax")
    quant_map = weights.get(f"{prefix}.weight.quant_map")
    nested_absmax = weights.get(f"{prefix}.weight.nested_absmax")
    nested_quant_map = weights.get(f"{prefix}.weight.nested_quant_map")

    # Reconstruire le dict quantized_stats attendu par from_prequantized
    quantized_stats = {
        "quant_state.bitsandbytes__nf4": quant_state_bytes,
        "absmax": absmax,
        "quant_map": quant_map,
    }
    if nested_absmax is not None:
        quantized_stats["nested_absmax"] = nested_absmax
        quantized_stats["nested_quant_map"] = nested_quant_map

    param = Params4bit.from_prequantized(
        data=qweight,
        quantized_stats=quantized_stats,
        device="cpu"
    )
    # Déquantifier vers float32
    return bnb.functional.dequantize_4bit(param.data, param.quant_state).float()


# ── Identifier toutes les couches base_layer quantifiées ─────────────────────
base_prefixes = set()
for k in saved_weights:
    if k.endswith(".base_layer.weight"):
        base_prefixes.add(k.replace(".weight", ""))

print(f"Couches à déquantifier : {len(base_prefixes)}")

# ── Déquantifier + merger LoRA pour chaque couche ────────────────────────────
merged_state_dict = {}

for prefix in base_prefixes:
    clean_key = prefix.replace(".base_layer", "") + ".weight"
    try:
        W = dequant_layer(prefix, saved_weights)
    except Exception as e:
        print(f"❌ Échec dequant {prefix}: {e}")
        continue

    # Chercher LoRA correspondant
    layer_base = prefix.replace(".base_layer", "")
    lora_a_key = f"{layer_base}.lora_A.default.weight"
    lora_b_key = f"{layer_base}.lora_B.default.weight"

    if lora_a_key in saved_weights and lora_b_key in saved_weights:
        A = saved_weights[lora_a_key].float()
        B = saved_weights[lora_b_key].float()
        W = W + (B @ A) * 2.0   # scaling = alpha/r = 32/16

    merged_state_dict[clean_key] = W

    # Bias — pas quantifié, copier direct
    bias_key = f"{prefix}.bias"
    if bias_key in saved_weights:
        merged_state_dict[clean_key.replace(".weight", ".bias")] = saved_weights[bias_key].float()

print(f"✅ {len(merged_state_dict)} tenseurs déquantifiés et mergés")

# ── Copier les poids non-quantifiés tels quels (embeddings, LayerNorm) ───────
for k, v in saved_weights.items():
    if "base_layer" not in k and "lora_" not in k and "nested" not in k and "quant" not in k and "absmax" not in k:
        merged_state_dict[k] = v.float()

print(f"Total final : {len(merged_state_dict)} tenseurs")

Couches à déquantifier : 73


f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\backends\cpu\ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\backends\cpu\ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ 146 tenseurs déquantifiés et mergés
Total final : 199 tenseurs


In [4]:
# ── Charger dans un modèle vierge et sauvegarder ─────────────────────────────
base_model = AutoModel.from_pretrained(
    "intfloat/multilingual-e5-base",
    dtype = torch.float32
)

missing, unexpected = base_model.load_state_dict(merged_state_dict, strict=False)
print(f"Missing : {len(missing)}  | Unexpected : {len(unexpected)}")
if missing:
    print("Missing keys (premiers 5):", missing[:5])

base_model.save_pretrained(EVAL_PATH, safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
tokenizer.save_pretrained(EVAL_PATH)

import shutil
for item in ["modules.json", "sentence_bert_config.json", "config_sentence_transformers.json", "1_Pooling", "2_Normalize"]:
    src = os.path.join("./e5-embedding-merged", item)
    if os.path.exists(src):
        dst = os.path.join(EVAL_PATH, item)
        shutil.copytree(src, dst, dirs_exist_ok=True) if os.path.isdir(src) else shutil.copy(src, dst)

print(f"✅ {EVAL_PATH}")

W0705 18:35:58.966000 6668 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0705 18:35:59.177000 6668 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Missing : 50  | Unexpected : 50
Missing keys (premiers 5): ['embeddings.LayerNorm.weight', 'embeddings.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight']


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ./e5-embedding-eval-clean


In [5]:
# ── Corriger le nommage legacy BERT → transformers moderne ───────────────────
renamed_state_dict = {}

for k, v in merged_state_dict.items():
    new_k = k.replace(".gamma", ".weight").replace(".beta", ".bias")
    renamed_state_dict[new_k] = v

print(f"Clés renommées : {sum(1 for k in merged_state_dict if k != k.replace('.gamma','.weight').replace('.beta','.bias'))}")

# ── Recharger avec les noms corrigés ──────────────────────────────────────────
base_model = AutoModel.from_pretrained(
    "intfloat/multilingual-e5-base",
    dtype = torch.float32
)

missing, unexpected = base_model.load_state_dict(renamed_state_dict, strict=False)
print(f"Missing : {len(missing)}  | Unexpected : {len(unexpected)}")
if missing:
    print("Missing (reste) :", missing[:10])
if unexpected:
    print("Unexpected (reste) :", unexpected[:10])

# Si Missing == 0 et Unexpected == 0 (ou juste position_ids) → parfait

Clés renommées : 50


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Missing : 0  | Unexpected : 0


In [6]:
# ── Sauvegarder la version corrigée ───────────────────────────────────────────
EVAL_PATH = "./e5-embedding-eval-clean"

base_model.save_pretrained(EVAL_PATH, safe_serialization=True)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
tokenizer.save_pretrained(EVAL_PATH)

import shutil, os
for item in ["modules.json", "sentence_bert_config.json", "config_sentence_transformers.json", "1_Pooling", "2_Normalize"]:
    src = os.path.join("./e5-embedding-merged", item)
    if os.path.exists(src):
        dst = os.path.join(EVAL_PATH, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy(src, dst)

print(f"✅ Modèle final propre : {EVAL_PATH}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modèle final propre : ./e5-embedding-eval-clean


In [7]:
from sentence_transformers import SentenceTransformer
import torch

st_eval = SentenceTransformer(
    "./e5-embedding-eval-clean",
    device       = "cuda",
    model_kwargs = {"torch_dtype": torch.bfloat16}
)

print(st_eval)
print(f"VRAM : {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Test de cohérence sémantique
test_q = st_eval.encode(["query: Quelle est la capitale de la France ?"])
test_p = st_eval.encode(["passage: Paris est la capitale de la France."])
test_n = st_eval.encode(["passage: Le bœuf bourguignon est un plat traditionnel."])

sim_pos = (torch.tensor(test_q) @ torch.tensor(test_p).T).item()
sim_neg = (torch.tensor(test_q) @ torch.tensor(test_n).T).item()

print(f"Sim pos : {sim_pos:.4f}")
print(f"Sim neg : {sim_neg:.4f}")
print(f"Margin  : {sim_pos - sim_neg:.4f}")

assert sim_pos > sim_neg, "❌ Le modèle ne rank pas correctement"
print("\n✅ Modèle validé — prêt pour MTEB")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
VRAM : 0.56 GB
Sim pos : 0.7718
Sim neg : 0.0337
Margin  : 0.7381

✅ Modèle validé — prêt pour MTEB


In [ ]:

all_task_names = [t.metadata.name for t in benchmark.tasks]

def already_done(task_name):
    return os.path.exists(os.path.join(OUTPUT_FOLDER, f"{task_name}.json"))

todo = [t for t in all_task_names if not already_done(t)]
done = [t for t in all_task_names if     already_done(t)]

print(f"✅ Déjà terminées : {len(done)}")
print(f"⏳ Restantes      : {len(todo)}")

all_scores = {}
for task_name in done:
    with open(os.path.join(OUTPUT_FOLDER, f"{task_name}.json")) as f:
        data = json.load(f)
    if "score" in data:
        all_scores[task_name] = data["score"]

# ── Évaluation tâche par tâche avec sauvegarde immédiate ─────────────────────
for i, task_name in enumerate(todo):
    vram = torch.cuda.memory_allocated() / 1e9
    print(f"[{i+1}/{len(todo)}] {task_name} | VRAM: {vram:.2f}GB ...", end=" ", flush=True)

    try:
        task    = mteb.get_tasks(tasks=[task_name], languages=["fra"])
        results = mteb.evaluate(
            st_eval,
            tasks         = task,
            encode_kwargs = {"batch_size": 32},
        )
        score = results[0].get_score()
        all_scores[task_name] = score

        with open(os.path.join(OUTPUT_FOLDER, f"{task_name}.json"), "w") as f:
            json.dump({"task": task_name, "score": score}, f, indent=2)

        print(f"✅ {score:.4f}")

    except torch.cuda.OutOfMemoryError:
        print(f"❌ OOM — retry batch_size=8")
        try:
            results = mteb.evaluate(st_eval, tasks=task, encode_kwargs={"batch_size": 8})
            score = results[0].get_score()
            all_scores[task_name] = score
            with open(os.path.join(OUTPUT_FOLDER, f"{task_name}.json"), "w") as f:
                json.dump({"task": task_name, "score": score}, f, indent=2)
            print(f"✅ {score:.4f} (batch=8)")
        except Exception as e2:
            print(f"❌ {e2}")
            with open(os.path.join(OUTPUT_FOLDER, f"{task_name}_ERROR.json"), "w") as f:
                json.dump({"task": task_name, "error": str(e2)}, f)

    except Exception as e:
        print(f"❌ {e}")
        with open(os.path.join(OUTPUT_FOLDER, f"{task_name}_ERROR.json"), "w") as f:
            json.dump({"task": task_name, "error": str(e)}, f)

    finally:
        gc.collect()
        torch.cuda.empty_cache()

# ── Résumé ────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
if all_scores:
    mean = sum(all_scores.values()) / len(all_scores)
    print(f"Score moyen MTEB(fra) : {mean:.4f}  ({len(all_scores)}/{len(all_task_names)} tâches)")
    df_r = pd.DataFrame([
        {"task": k, "score": round(v, 4)}
        for k, v in sorted(all_scores.items(), key=lambda x: -x[1])
    ])
    print(df_r.to_string(index=False))
    df_r.to_csv(os.path.join(OUTPUT_FOLDER, "summary.csv"), index=False)
    print(f"\n💾 {OUTPUT_FOLDER}/summary.csv")

[mteb.get_tasks|WARNING]Skipping task 'Assin2STS': no subsets match the language filter ['pt'].
[mteb.get_tasks|WARNING]Skipping task 'SICK-BR-STS': no subsets match the language filter ['pt'].
[mteb.get_tasks|WARNING]Skipping task 'STSBenchmarkMultilingualSTS': no subsets match the language filter ['pt'].
[mteb.get_tasks|WARNING]Skipping task 'MassiveIntentClassification': no subsets match the language filter ['pt'].
[mteb.get_tasks|WARNING]Skipping task 'BrazilianToxicTweetsClassification': no subsets match the language filter ['pt'].
[mteb.get_tasks|WARNING]Skipping task 'HateSpeechPortugueseClassification': no subsets match the language filter ['pt'].
[mteb.get_tasks|WARNING]Skipping task 'MultiLongDocReranking': no subsets match the language filter ['pt'].
[mteb.get_tasks|WARNING]Skipping task 'WikipediaRerankingMultilingual': no subsets match the language filter ['pt'].
[mteb.get_tasks|WARNING]Skipping task 'XGlueWPRReranking': no subsets match the language filter ['pt'].
[mteb.g

✅ Déjà terminées : 0
⏳ Restantes      : 25
[1/25] AmazonReviewsClassification | VRAM: 2.27GB ... 

Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 0.4535
[2/25] MasakhaNEWSClassification | VRAM: 2.27GB ... 

Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

...

✅ 0.7060
[24/25] SummEvalFr | VRAM: 2.27GB ... 

Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 0.2672
[25/25] STS22 | VRAM: 2.27GB ... 

Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 0.5924

Score moyen MTEB(fra) : 0.5060  (25/25 tâches)
                         task  score
     MTOPDomainClassification 0.9759
     MTOPIntentClassification 0.8612
MassiveScenarioClassification 0.8364
  MassiveIntentClassification 0.7667
  STSBenchmarkMultilingualSTS 0.7060
                       SICKFr 0.6859
    MasakhaNEWSClassification 0.6462
                        STS22 0.5924
              SyntecReranking 0.5901
      PawsXPairClassification 0.5875
            AlloprofReranking 0.5300
               BSARDRetrieval 0.5045
        AlloProfClusteringP2P 0.4973
        AlloProfClusteringS2S 0.4741
           MLSUMClusteringP2P 0.4556
  AmazonReviewsClassification 0.4535
           MLSUMClusteringS2S 0.4382
              SyntecRetrieval 0.4213
     MasakhaNEWSClusteringS2S 0.4102
                XPQARetrieval 0.3363
                   SummEvalFr 0.2672
     MasakhaNEWSClusteringP2P 0.2620
             HALClusteringS2S 0.2162
            AlloprofRetrieval 0.0893
             Minta

In [1]:
model = mteb.get_model("./e5-embedding-eval-clean")  # ou ton checkpoint final
mteb.evaluate(model, tasks=benchmark)

NameError: name 'mteb' is not defined

In [32]:
# Vérifier si la config prompts existe dans le modèle original
import json
with open("./e5-embedding-merged/config_sentence_transformers.json") as f:
    print(json.load(f))

# Et dans le modèle reconstruit
with open("./e5-embedding-eval-clean/config_sentence_transformers.json") as f:
    print(json.load(f))

{'__version__': {'pytorch': '2.12.1+cu130', 'sentence_transformers': '5.6.0', 'transformers': '5.5.0'}, 'default_prompt_name': None, 'model_type': 'SentenceTransformer', 'prompts': {'document': '', 'query': ''}, 'similarity_fn_name': 'cosine'}
{'__version__': {'pytorch': '2.12.1+cu130', 'sentence_transformers': '5.6.0', 'transformers': '5.5.0'}, 'default_prompt_name': None, 'model_type': 'SentenceTransformer', 'prompts': {'document': '', 'query': ''}, 'similarity_fn_name': 'cosine'}


In [33]:
from sentence_transformers import SentenceTransformer
orig = SentenceTransformer("intfloat/multilingual-e5-base")
print(orig.prompts)
# Si non vide → confirme l'hypothèse du double-préfixage

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'query': '', 'document': ''}


In [34]:
# Test direct — comment le modèle voit un texte sans prefix (ce que MTEB envoie probablement)
no_prefix_q = st_eval.encode(["Quelle est la capitale de la France ?"])  # SANS "query: "
no_prefix_p = st_eval.encode(["Paris est la capitale de la France."])     # SANS "passage: "
no_prefix_n = st_eval.encode(["Le bœuf bourguignon est un plat traditionnel."])

sim_pos_noprefix = (torch.tensor(no_prefix_q) @ torch.tensor(no_prefix_p).T).item()
sim_neg_noprefix = (torch.tensor(no_prefix_q) @ torch.tensor(no_prefix_n).T).item()

print(f"SANS préfixe — Sim pos : {sim_pos_noprefix:.4f} | Sim neg : {sim_neg_noprefix:.4f} | Margin : {sim_pos_noprefix-sim_neg_noprefix:.4f}")

# Comparer avec la version AVEC préfixe (déjà testée : margin 0.7381)
print(f"AVEC préfixe (déjà connu) — Margin : 0.7381")

SANS préfixe — Sim pos : 0.8415 | Sim neg : 0.0654 | Margin : 0.7761
AVEC préfixe (déjà connu) — Margin : 0.7381
